# Lab 3: PyTorch for Cat vs Dog Faces

In this notebook, you will train a small **binary image classifier** with PyTorch and compare it against your NumPy baseline.

We will focus on the core training pipeline:

- dataset and dataloader
- image tensors
- a simple MLP model
- loss and optimizer
- training and evaluation loops
- comparison with the Lab 1 baseline

**Questions in this lab**

1. Load metadata and map labels to integers
2. Build a dataset that returns tensors
3. Create train, validation, and test DataLoaders
4. Inspect one mini-batch
5. Define a simple MLP classifier
6. Set up loss, optimizer, and device
7. Complete one training epoch
8. Evaluate the model and compare it with the NumPy baseline


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from lab_utils.visualization import (
    extract_feature_maps,
    plot_feature_maps_like_reference,
    plot_training_history,
    show_tensor_batch,
)

DATA_ROOT = Path("data/cats_dogs_faces_small")
METADATA_PATH = DATA_ROOT / "metadata.csv"
NUMPY_PRED_PATH = Path("artifacts/lab1_numpy_predictions.csv")

LABELS = ("cat", "dog")
SPLITS = ("train", "val", "test")

def build_metadata_from_folders(data_root: Path) -> pd.DataFrame:
    rows = []
    for split in SPLITS:
        for label in LABELS:
            label_dir = data_root / split / label
            for path in sorted(label_dir.glob("*.jpg")) + sorted(label_dir.glob("*.png")):
                with Image.open(path) as image:
                    image = image.convert("RGB")
                    width, height = image.size
                rows.append(
                    {
                        "filepath": str(path.relative_to(data_root)),
                        "label": label,
                        "split": split,
                        "width": width,
                        "height": height,
                    }
                )
    return pd.DataFrame(rows)

if not DATA_ROOT.exists():
    raise FileNotFoundError(
        "Dataset not found. Place the prepared subset at data/cats_dogs_faces_small/."
    )

if METADATA_PATH.exists():
    df = pd.read_csv(METADATA_PATH)
else:
    df = build_metadata_from_folders(DATA_ROOT)

print(df.head())
print(df["split"].value_counts())


## Question 1: Turn string labels into integer labels

Create:

- a dictionary `label_to_index`
- a new column `label_id`

Use `cat -> 0` and `dog -> 1`.


In [ ]:
# TODO: build the mapping and the new label_id column
label_to_index = {}
train_df = df[df["split"] == "train"].copy()
val_df = df[df["split"] == "val"].copy()
test_df = df[df["split"] == "test"].copy()

print(label_to_index)
print(train_df.head())

assert set(label_to_index.keys()) == {"cat", "dog"}, "Both classes should appear in the mapping."


## Question 2: Build a dataset that returns tensors

Complete:

- `image_to_tensor`
- `CatsDogsDataset.__getitem__`

Your dataset should return:

- an image tensor with shape `(3, 64, 64)`
- an integer label tensor


In [ ]:
def image_to_tensor(path: Path) -> torch.Tensor:
    # TODO: open the image, convert to RGB, normalize to [0, 1],
    # and permute to channel-first format (C, H, W)
    raise NotImplementedError("Convert an image file into a float tensor.")

class CatsDogsDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, data_root: Path):
        self.frame = frame.reset_index(drop=True)
        self.data_root = data_root

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, index: int):
        row = self.frame.iloc[index]
        # TODO: load the image tensor and return (image_tensor, label_tensor)
        raise NotImplementedError("Complete __getitem__.")

train_dataset = CatsDogsDataset(train_df, DATA_ROOT)
first_image, first_label = train_dataset[0]
print(first_image.shape, first_image.dtype, first_label)


## Question 3: Create DataLoaders

Build three DataLoaders:

- training loader with `shuffle=True`
- validation loader with `shuffle=False`
- test loader with `shuffle=False`

Use a batch size of `32`.


In [ ]:
BATCH_SIZE = 32

# TODO: create val_dataset, test_dataset, and the three dataloaders
train_loader = None
val_loader = None
test_loader = None

train_loader, val_loader, test_loader


## Question 4: Inspect one mini-batch

Pull one batch from the training loader and verify:

- image batch shape
- label batch shape
- image dtype
- label dtype


In [ ]:
if train_loader is None:
    raise ValueError("Complete Question 3 before inspecting a batch.")

batch_images, batch_labels = next(iter(train_loader))

print("Image batch:", batch_images.shape, batch_images.dtype)
print("Label batch:", batch_labels.shape, batch_labels.dtype)

assert batch_images.ndim == 4, "Batches of images should have shape (B, C, H, W)."
assert batch_images.shape[1] == 3, "Color images should have 3 channels."

show_tensor_batch(
    batch_images[:8].cpu().numpy(),
    batch_labels[:8].cpu().numpy(),
    class_names=LABELS,
    max_items=8,
    ncols=4,
)


## Question 5: Define a simple MLP classifier

Complete the model below.

Suggested architecture:

- flatten
- linear layer to 128 hidden units
- ReLU
- linear layer to 2 output logits


In [ ]:
class CatsDogsMLP(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: define self.network as an nn.Sequential model
        raise NotImplementedError("Define the MLP layers here.")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)

model = CatsDogsMLP()
example_logits = model(batch_images[:4])
print(example_logits.shape)

assert example_logits.shape == (4, 2), "The MLP should output two logits per image."


## Question 6: Set up the training ingredients

Choose:

- a device (`cuda` if available, otherwise `cpu`)
- a loss function
- an optimizer

Use `CrossEntropyLoss` and `Adam` for this lab.


In [ ]:
# TODO: move the model to the chosen device and create criterion + optimizer
device = None
criterion = None
optimizer = None

print("Using device:", device)


## Question 7: Complete one training epoch

Fill in the missing logic for:

- zeroing gradients
- forward pass
- loss computation
- backward pass
- optimizer step
- batch accuracy tracking


In [ ]:
def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    device: torch.device,
) -> tuple[float, float]:
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        # TODO: complete the training step
        raise NotImplementedError("Implement the training loop for one epoch.")

    average_loss = total_loss / total_examples
    average_accuracy = total_correct / total_examples
    return average_loss, average_accuracy


## Question 8: Evaluate the model and compare it with the NumPy baseline

Complete the evaluation loop, train for a few epochs, and compare test accuracy with Lab 1.

Reflection prompts:

1. Did the MLP outperform the NumPy baseline?
2. Did the model start to overfit?
3. What would you try next: more data, a CNN, augmentation, or tuning?


In [ ]:
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> tuple[float, float]:
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            # TODO: compute logits, loss, and accuracy for evaluation
            raise NotImplementedError("Implement the evaluation loop.")

    average_loss = total_loss / total_examples
    average_accuracy = total_correct / total_examples
    return average_loss, average_accuracy

EPOCHS = 5
history = []

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)
    history.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
        }
    )
    print(
        f"Epoch {epoch}: "
        f"train_loss={train_loss:.4f}, train_acc={train_acc:.3f}, "
        f"val_loss={val_loss:.4f}, val_acc={val_acc:.3f}"
    )

test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print(f"Test accuracy (PyTorch MLP): {test_acc:.3f}")

if NUMPY_PRED_PATH.exists():
    numpy_pred_df = pd.read_csv(NUMPY_PRED_PATH)
    numpy_baseline_acc = numpy_pred_df["correct_numpy"].mean()
    print(f"Test accuracy (NumPy baseline): {numpy_baseline_acc:.3f}")
else:
    print("NumPy baseline predictions not found. Run Lab 1 if you want a direct comparison.")

plot_training_history(history)

history


## Optional Visualization: Feature Maps Like the AlexNet Notebook

The main lab uses an MLP, so it does **not** produce spatial feature maps.
To explore convolutional activations, use a tiny CNN and the shared helper below.

The visualization utility is intentionally styled like the tiled activation-grid view from the referenced AlexNet notebook:

- one tile per channel
- `viridis` colormap
- light gaps between maps
- fixed `vmin` and `vmax` controls so maps are easier to compare

If you train your own CNN later, replace `cnn_for_viz.features` with your trained feature extractor.


In [ ]:
class TinyCNNForViz(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
        )
        self.classifier = nn.Sequential(
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(32 * 16 * 16, 64),
            nn.ReLU(),
            nn.Linear(64, 2),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        return self.classifier(x)

feature_map_device = device if device is not None else torch.device("cpu")
cnn_for_viz = TinyCNNForViz().to(feature_map_device)
example_image, _ = train_dataset[0]

first_block_maps = extract_feature_maps(
    cnn_for_viz.features,
    example_image,
    layer_up_to=2,
    device=feature_map_device,
)

plot_feature_maps_like_reference(
    first_block_maps,
    grid_size=(4, 4),
    vmin=-0.7,
    vmax=0.5,
    title="Tiny CNN feature maps after the first Conv + ReLU block",
)


## Optional extension

If you finish early, replace the MLP with a tiny CNN:

- `Conv2d(3, 16, kernel_size=3, padding=1)`
- `ReLU`
- `MaxPool2d(2)`
- `Conv2d(16, 32, kernel_size=3, padding=1)`
- `ReLU`
- `MaxPool2d(2)`
- `Flatten`
- `Linear(...)`
- `Linear(..., 2)`

Then compare the CNN with the MLP on validation accuracy and reuse
`plot_feature_maps_like_reference(...)` to inspect how the first and second
convolution blocks respond to the same image.
